In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path = [
    '/usr/lib/python36.zip',
    '/usr/lib/python3.6',
    '/usr/lib/python3.6/lib-dynload',
    '/usr/local/lib/python3.6/dist-packages',
    '/usr/lib/python3/dist-packages',
    '/usr/local/lib/python3.6/dist-packages/IPython/extensions',
]

sys.path.append('/projects/ngs_eco/users/kmvr819/PODS/GeMinAI/clinical_transformer/')
sys.path.append('/projects/ngs_eco/users/kmvr819/PODS/GeMinAI/samecode/')
sys.path.append('/wscratch/ai_data_center/ods_eds_aidc_hub/')

In [ ]:
import matplotlib
import pandas as pd
matplotlib.rcParams['figure.dpi'] = 256
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 500)
%config InlineBackend.figure_format = 'svg'

In [ ]:
from samecode.logger.mlflow import Logger
from samecode.random import set_seed

import pandas as pd

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
from xai.models.explainer import SurvivalExtractor
from samecode.plot.props import BOXPLOT

In [ ]:
import seaborn as sns
from samecode.plot.pyplot import subplots
from samecode.random import set_seed

# **# Peripheral blood**

Get Embeddings from RFM (RNA Foundation Model)

In [ ]:
from xai.models.explainer import TransformerSelfSupervisedEvaluator
from xai.models.explainer import selfsupervision_attention_scores
from xai.models import load_transformer

In [ ]:
import umap
import pickle
import yaml

In [ ]:
# recount3 = pd.read_excel('/wscratch/ai_data_center/primary_data_center/Data_catalog_v1.1.xlsx', engine='openpyxl', sheet_name='Recount3')
# human = recount3.query('organism == "Human"').query('RNAseq_Type == "bulkRNA-seq"').reset_index(drop=True)
# studies = human['study identifiers'].drop_duplicates()

In [ ]:
data = pd.read_csv('~/GenAIProgram/foundation_models/modeling/clinical_transformer/bulk_rna/pan_tissue/01-FGEs+IO/01-GTEx+TCGA+RECOUNT3/data/gtex+tcga+recount3+FGEs+io.csv', low_memory=False)
# data = data[data.tissue.isin(studies)].reset_index(drop=True)
# data = data.loc[data.median(axis=1) != 0, :].reset_index(drop=True)

data = data[data.source_data.isin(['GTEx', 'TCGA'])]
data.shape

In [ ]:
data = data.dropna().reset_index(drop=True)

In [ ]:
data = data.query('tissue == "whole_blood"').reset_index(drop=True)

In [ ]:
data.shape

In [ ]:
model_config = yaml.safe_load(open('../environment/model_config.yaml'))
trainer = load_transformer(model_config['model']['path'], model_config['model']['run'], epoch=model_config['model']['epoch'])

In [ ]:
transformed_data = trainer.data_converter.transform(data).reset_index(drop=True)

In [ ]:
set_seed(0)
evaluator = TransformerSelfSupervisedEvaluator(model=trainer)
_, outputs, features, patient_ids, iters, attention_scores, _ = selfsupervision_attention_scores(
    transformed_data, evaluator, iterations=1,
    sample_id='patient_id', batch_size=data.shape[0],
    return_attentions=True
)

In [ ]:
outputs.shape

## Extract Embeddings

We extract the embeddings from the last transformer layer. We selected the ```<cls>``` toke as it is not attached to any task and we consider it as our patient embeddings. Similar to DINO use of cls token (https://arxiv.org/pdf/2104.14294)

In [ ]:
from samecode.plot.pyplot import subplots
from samecode.survival.plot import KMPlot
from samecode.plot.pretty import heatmap
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, SpectralClustering, AffinityPropagation, DBSCAN, AgglomerativeClustering
import pickle
import numpy as np

In [ ]:
colors = ['#dabfff', '#907ad6', '#a44200', '#2c2a4a']

In [ ]:
feature = '<cls>'

embeddings = outputs[features == feature]
pts = patient_ids[features == feature]

emb_names = ['E{}'.format(i) for i in range(outputs.shape[1])]
emb = pd.DataFrame(embeddings, columns=emb_names)
emb['patient_id'] = pts

df = pd.merge(data, emb, on='patient_id')

In [ ]:
df.shape

## Automatize selection of best clusters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, silhouette_samples
from sklearn.mixture import GaussianMixture
from scipy.spatial.distance import cdist
from sklearn.cluster import AgglomerativeClustering

In [ ]:
# df.query('tissue == "whole_blood"')

In [ ]:
# Elbow Method
wcss = []
silhouette_scores = []
chs = []
K = range(2, 50)

for k in K:
#     print(k)
    clustering = KMeans(n_clusters=k, random_state=0).fit(df.query('tissue == "whole_blood"')[emb_names])
    labels = clustering.labels_

    wcss.append(clustering.inertia_)
    silhouette_scores.append(silhouette_score(df.query('tissue == "whole_blood"')[emb_names], labels))
    chs.append(calinski_harabasz_score(df.query('tissue == "whole_blood"')[emb_names], labels))

In [ ]:
# pickle.dump([K, wcss, silhouette_scores, chs], open('../data/k-clusters.pk', 'wb'))

In [ ]:
# [K, wcss, silhouette_scores, chs] = pickle.load(open('../data/k-clusters.pk', 'rb'))

In [ ]:
f, axs = subplots(cols=1, rows=1, w=5, h=4, return_f=True)
sns.lineplot(x=K, y=silhouette_scores, ax=axs[0], markers=["bx+"])
sns.scatterplot(x=K, y=silhouette_scores, ax=axs[0], markers=["bx-"]);

axs[0].set_title('Silhouette')

sns.despine()

## Save best N for clustering

In [ ]:
kmeans = KMeans(random_state=0, n_clusters=n).fit(df.query('tissue == "whole_blood"')[emb_names])

# we lock the clusters here:
pickle.dump(kmeans, open(
    '../data/{}.clusters-{}.pk'.format(feature, n),
    'wb'
))

## Load kmeans

In [ ]:
n = 3

In [ ]:
kmeans = pickle.load(
            open('../data/{}.clusters-{}.pk'.format(feature, n), 'rb')
        )

df['cluster'] = kmeans.predict(df[emb_names])

# Analysis

In [ ]:
fts = [
    # Angiogenesis Fibroblasts
    'Endothelium', 'Cancer-associated fibroblasts', 'Matrix', 'Matrix remodeling',  'Angiogenesis',
    # Pro-tumor Immune infiltrate
    'Granulocyte traffic', 'Neutrophil signature', 'Macrophage and DC traffic', 'Tumor-associated Macrophages', 'Myeloid cells traffic', 'Immune Suppression by Myeloid Cells',   'CD1c', 'TLS_Chemokine', 'M1 signature',
    # Anti tumor immune infiltrate
     'MHC_I', 'MHC_II', 'Antitumor cytokines', 'Protumor cytokines', 'Checkpoint molecules','Co-activation molecules','B_cell',
    'Th2 signature', 'Treg and Th2 traffic', 'Treg',
    'NK', 'Effector cells', 'T cells',  'Th1 signature',  'Effector cell traffic',  'T_effector', 'IFNG',  'T_agonist',  'TLS_TFH',
    # Cancer related
    'Tumor proliferation rate', 'EMT signature',
]

In [ ]:
# f, axs = subplots(cols=1, rows=1, w=8, h=4, return_f=True)
# heatmap(df, axs[0], groupby='cluster', y_cut=[8, 15, 33], columns=fts, vmin=-2, vmax=2)

In [ ]:
matplotlib.rcParams['figure.dpi'] = 256
%config InlineBackend.figure_format = 'png'

tissues = ['whole_blood', ]

f, axs = subplots(cols=1, rows=len(tissues), w=8, h=3.5*len(tissues), return_f=True)
for ix, tissue in enumerate(tissues):

    dfi = df.query('tissue=="{}"'.format(tissue))
    heatmap(dfi, axs[ix], groupby='cluster', y_cut=[5, 14, 33], columns=fts, vmin=-2, vmax=2)
    axs[ix].set_title('{} (N={})'.format(tissue, dfi.shape[0]))

# tSNE and projection

In [ ]:
from sklearn.decomposition import PCA, SparsePCA
from sklearn.manifold import TSNE

In [ ]:
sdf = df.query('tissue == "whole_blood"').reset_index(drop=True)

In [ ]:
reducer = PCA(
    n_components=2,
)

In [ ]:
sdf[['P1', 'P2']] = reducer.fit_transform(sdf[emb_names])

In [ ]:
from samecode.plot.pyplot import clear_plot

palette=['#D90479', '#8C0F61', '#F29F05', 'darkblue']

%config InlineBackend.figure_format = 'png'
f, axs = subplots(cols=1, rows=1, w=8, h=6, return_f=True)

sns.kdeplot(data=sdf, x='P1', y='P2', hue='cluster',palette=palette , levels=2, zorder=0, fill=True, alpha=0.5, ax=axs[0])
sns.kdeplot(data=sdf, x='P1', y='P2', hue='cluster', palette=palette, levels=2, zorder=0, fill=False, alpha=1, ax=axs[0])
sns.scatterplot(data=sdf, x='P1', y='P2', hue='cluster', palette=palette, zorder=1, ax=axs[0])
axs[0].set_title('Peripheral blood (GTEx): immune clusters')

# clear_plot(axs[0])
# axs[0].tick_params(axis = "x", which = "both", bottom = False, top = False)
# axs[0].set_xlabel('')
sns.despine()


# Interpretability - Post Attention

In [ ]:
fts = [
    # Angiogenesis Fibroblasts
    'Endothelium', 'Cancer-associated fibroblasts', 'Matrix', 'Matrix remodeling',  'Angiogenesis',
    # Pro-tumor Immune infiltrate
    'Granulocyte traffic', 'Neutrophil signature', 'Macrophage and DC traffic', 'Tumor-associated Macrophages', 'Myeloid cells traffic', 'Immune Suppression by Myeloid Cells',   'CD1c', 'TLS_Chemokine', 'M1 signature',
    # Anti tumor immune infiltrate
     'MHC_I', 'MHC_II', 'Antitumor cytokines', 'Protumor cytokines', 'Checkpoint molecules','Co-activation molecules','B_cell',
    'Th2 signature', 'Treg and Th2 traffic', 'Treg',
    'NK', 'Effector cells', 'T cells',  'Th1 signature',  'Effector cell traffic',  'T_effector', 'IFNG',  'T_agonist',  'TLS_TFH',
    # Cancer related
    'Tumor proliferation rate', 'EMT signature',
]

In [ ]:
feature = '<cls>'
attention = attention_scores[(attention_scores.source == feature) | (attention_scores.target ==feature)].reset_index(drop=True)
attention['target'] = [i.target if i.source == feature else i.source for ix,i in attention.iterrows()]
attention['source'] = feature
attention = attention.pivot_table(index='id_', columns='target', values='score').reset_index()

In [ ]:
attention = pd.merge(attention, df[['cluster', 'patient_id']], left_on='id_', right_on='patient_id')

In [ ]:
from sklearn.preprocessing import RobustScaler
rs = RobustScaler().fit(attention[fts])

In [ ]:
# attention[fts] = (attention[fts] - np.min(attention[fts])) / (np.max(attention[fts]) - np.min(attention[fts]))
attention[fts] = (attention[fts] - np.mean(attention[fts])) / (np.std(attention[fts]))
attention[fts] = 1 / (1 + np.exp(-attention[fts] / 0.1))
# attention[fts] = rs.transform(attention[fts])

In [ ]:
%config InlineBackend.figure_format = 'png'
f, axs = subplots(cols=2, rows=1, w=12, h=4, return_f=True)

ycut = [5, 14, 33]
# ycut = [14, 25, 26, 28]
# ycut = []

heatmap(df, axs[0], groupby='cluster',  y_cut=ycut, columns=fts, vmin=-2, vmax=2)
heatmap(attention, axs[1], groupby='cluster', y_cut=ycut, columns=fts, vmin=0, vmax=1)

axs[0].set_xlabel('Expression Scores')
axs[1].set_xlabel('Relative Cosine Similarity');

### Predict differences between cluster 1 and 2

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    random_state=0,
).fit(df[nfts].dropna(), df.dropna()['cluster'])

In [ ]:
nfts=['Th1 signature', 'B_cell', 'T_effector', 'Co-activation molecules',]

In [ ]:
imp = pd.DataFrame([clf.feature_importances_], columns=nfts).mean().sort_values().reset_index()
imp.columns = ['variable', 'score']
imp

In [ ]:
df['rf_cluster'] = clf.predict(df[nfts])
df.value_counts(['cluster', 'rf_cluster']).reset_index()

# Sanity check: simple linear regression can recover in the training set the data from the clustering assignment.

In [ ]:
%config InlineBackend.figure_format = 'svg'



variables = imp.query('score != 0').variable.values

boxprops = dict(linestyle='-', linewidth=1, edgecolor='black')
flierprops = dict(marker='o', markerfacecolor='black', markersize=5,markeredgecolor='none')
medianprops = dict(linestyle='-', linewidth=1, color='black')

width=0.5

f, axs = subplots(cols=4, rows=1, w=9, h=2.3, return_f=True)

dfx = df.copy()#.rename(columns={'B_cell': 'B cells',})

sns.barplot(data=dfx.melt(id_vars='cluster', value_vars=variables).query('cluster == 0'), y='variable', x='value', order=variables, color=palette[0], ax=axs[0], )
sns.barplot(data=dfx.melt(id_vars='cluster', value_vars=variables).query('cluster == 1'), y='variable', x='value', order=variables, color=palette[1], ax=axs[1], )
sns.barplot(data=dfx.melt(id_vars='cluster', value_vars=variables).query('cluster == 2'), y='variable', x='value', order=variables, color=palette[2], ax=axs[2], )
sns.barplot(data=dfx.melt(id_vars='cluster', value_vars=variables).query('cluster == 3'), y='variable', x='value', order=variables, color=palette[3], ax=axs[3], )

for ix in range(4):
    axs[ix].set_xlim([-2, 2])
    label = ''
    if ix == 1:
        label = 'Signature Expression Score'
    axs[ix].set_xlabel(label)
    axs[ix].set_ylabel('')
    axs[ix].tick_params(axis = "y", which = "both", left = False, right = False)
    axs[ix].spines['left'].set_visible(False)
    axs[ix].spines['right'].set_visible(False)
    axs[ix].spines['top'].set_visible(False)
    axs[ix].spines['bottom'].set_visible(True)
    if ix > 0:
        axs[ix].set_yticklabels([])
#     else:
#         axs[ix].set_yticklabels(['B cells', 'Neutrophils', 'Th2', 'MDSC', 'Effector cell traffic', 'IFNG', 'T Effector', 'NK', 'Th1', "Granulocyte traffic"])

# sns.despine()

# Analyze Immune lineage signatures

In [ ]:
lineage = pd.read_csv(
    '~/GenAIProgram/datasets/ID-2024-08-23-21-19-22-380066/data/gtex+tcga+ID-2024-08-23-17-48-04-818673+lineages.csv'
)

In [ ]:
lineage

In [ ]:
lineage = pd.merge(lineage, df[['patient_id', 'cluster']], on='patient_id')

In [ ]:
matplotlib.rcParams['figure.dpi'] = 256
%config InlineBackend.figure_format = 'png'

fts = ['Granulocytes', 'Monocytes', 'Dendritic', 'NK-cells', 'B-cells', 'T-cells']

tissues = ['whole_blood']

f, axs = subplots(cols=1, rows=len(tissues), w=8, h=2*len(tissues), return_f=True)
for ix, tissue in enumerate(tissues):

    dfi = lineage.query('tissue=="{}"'.format(tissue))
    heatmap(dfi, axs[ix], groupby='cluster', y_cut=[3,4], columns=fts, vmin=-2, vmax=2)
    axs[ix].set_title('{} (N={})'.format(tissue, dfi.shape[0]))

## Extra

In [ ]:
# f, axs = subplots(cols=2, rows=1, w=12, h=4, return_f=True)

# heatmap(
#     pd.concat([df[fts] * attention[fts], df['cluster']], axis=1),
#     axs[0], groupby='cluster', y_cut=ycut, columns=fts, vmin=-2, vmax=2
# )

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import colors

cmap = plt.cm.seismic

direction = df.groupby('cluster').median()[fts].reset_index().melt(id_vars='cluster', value_vars=fts)
normalizer = plt.Normalize(vmin=min(direction.value), vmax=max(direction.value))
mapped_colors = cmap(normalizer(direction.value))
direction['color'] = [colors.rgb2hex(c) for c in mapped_colors]

In [ ]:
from samecode.plot.network import bipartite_graph

In [ ]:
att = attention.groupby('cluster').median().reset_index().melt(id_vars='cluster', value_vars=fts)
# att = attention.reset_index().melt(id_vars='cluster', value_vars=fts)

In [ ]:
att = pd.merge(att, direction[['cluster', 'variable', 'color']], on=['cluster', 'variable'])

In [ ]:
# sns.kdeplot(att.value)

In [ ]:
# att = att[(att.value > 0.25) | (att.value < -0.25)].reset_index(drop=True)

In [ ]:
att['value'] = 2 * (att.value) + 0.01

In [ ]:
# att = pd.merge(att, ftsg, on='variable')

In [ ]:
# ftsg.sort_values('group').reset_index(drop=True).drop_duplicates('group')

In [ ]:
%config InlineBackend.figure_format = 'png'

In [ ]:
import matplotlib.patches as patches

ignoref = ['Tumor proliferation rate', 'EMT signature', 'Endothelium', 'Cancer-associated fibroblasts', 'Matrix', 'Matrix remodeling',  'Angiogenesis',]
left_nodes = [i for i in fts if not i in ignoref]

f, axs = subplots(cols=4, rows=1, w=20, h=6, return_f=True)
for ix, node in enumerate([0, 1, 2, 3]):
    edges = att[att.variable.isin(left_nodes) & att.cluster.isin([node])][['variable', 'cluster', 'value', 'color']].values
    bipartite_graph(
        left_nodes, [node], edges, {}, ax=axs[ix], alpha = 1, pos_color='white', label_color='black',
        rename={},
    )

### PCA distribution


In [ ]:
%config InlineBackend.figure_format = 'svg'

from samecode.plot.pyplot import clear_plot

variables = fts[3:-2]

f, axs = subplots(cols=4, rows=1, w=15, h=6, return_f=True)

d1 = attention.melt(id_vars='cluster', value_vars=variables).query('cluster == 0')
order = d1.groupby('variable').median().sort_values('value', ascending=False).reset_index().variable.values
sns.barplot(data=d1, y='variable', x='value', order=order, color='darkred', ax=axs[0], )

d1 = attention.melt(id_vars='cluster', value_vars=variables).query('cluster == 1')
order = d1.groupby('variable').median().sort_values('value', ascending=False).reset_index().variable.values
sns.barplot(data=d1, y='variable', x='value', order=order, color='lightblue', ax=axs[1], )

d1 = attention.melt(id_vars='cluster', value_vars=variables).query('cluster == 2')
order = d1.groupby('variable').median().sort_values('value', ascending=False).reset_index().variable.values
sns.barplot(data=d1, y='variable', x='value', order=order, color='gray', ax=axs[2], )

d1 = attention.melt(id_vars='cluster', value_vars=variables).query('cluster == 3')
order = d1.groupby('variable').median().sort_values('value', ascending=False).reset_index().variable.values
sns.barplot(data=d1, y='variable', x='value', order=order, color='green', ax=axs[3], )


# clear_plot(axs[1])
# clear_plot(axs[2])
# clear_plot(axs[3])

for ix in range(4):
    axs[ix].set_xlabel('')
    axs[ix].set_ylabel('')
    axs[ix].set_xlim([0, 1])

sns.despine()

### Global attention

In [ ]:
global_attention = attention_scores[(attention_scores.source == feature) | (attention_scores.target ==feature)].reset_index(drop=True)
global_attention.groupby('target').median().abs().reset_index().sort_values('score')

In [ ]:
# ftsg.query('group == 0').variable.values

In [ ]:
crr = pd.DataFrame(np.tril(np.abs(np.corrcoef(attention[fts].T)), -1), columns=fts)
crr['variables'] = fts

In [ ]:
crr = crr.melt(id_vars='variables', value_vars=fts)
crr.columns = ['source', 'target', 'score']

In [ ]:
ntx = crr.query('score > 0.5').query('score < 0.999').sort_values('score')

In [ ]:
ntx

### what features discriminate cluster 2 of 3

In [ ]:
recount3 = pd.read_excel('/wscratch/ai_data_center/primary_data_center/Data_catalog_v1.1.xlsx', engine='openpyxl', sheet_name='Recount3')

In [ ]:
human = recount3.query('organism == "Human"').query('RNAseq_Type == "bulkRNA-seq"').reset_index(drop=True)
blood = human[human['combined source'].astype(str).str.contains('blood|Blood|peripheral|Peripheral')]

studies = blood['study identifiers'].drop_duplicates()

In [ ]:
data = pd.read_csv('~/fm_hub/foundation_models/modeling/ID-2024-08-12-23-26-21-303517/data/gtex+tcga+recount3+FGEs+io.csv', low_memory=False)
recount = data[data.tissue.isin(studies)].reset_index(drop=True)
# data['sample_class'] = 'normal'
# data['tissue'] = 'whole_blood'
recount.shape

In [ ]:
recount = recount.loc[recount.median(axis=1) != 0, :].reset_index(drop=True)
recount.shape

In [ ]:
transformed_data = trainer.data_converter.transform(recount).reset_index(drop=True)

In [ ]:
set_seed(0)
evaluator = TransformerSelfSupervisedEvaluator(model=trainer)
_, outputs, features, patient_ids, iters, _, _ = selfsupervision_attention_scores(
    transformed_data, evaluator, iterations=1,
    sample_id='patient_id', batch_size=data.shape[0],
    return_attentions=False
)

In [ ]:
feature = '<cls>'

embeddings = outputs[features == feature]
pts = patient_ids[features == feature]

emb_names = ['E{}'.format(i) for i in range(outputs.shape[1])]
emb = pd.DataFrame(embeddings, columns=emb_names)
emb['patient_id'] = pts

recount_embeddings = pd.merge(recount, emb, on='patient_id')

In [ ]:
recount_embeddings['cluster'] = kmeans.predict(recount_embeddings[emb_names])

In [ ]:
f, axs = subplots(cols=1, rows=1, w=8, h=4, return_f=True)
heatmap(recount_embeddings, axs[0], groupby='cluster', y_cut = [], columns=fts, vmin=-2, vmax=2)

In [ ]:
recount_embeddings

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(
    random_state=0, max_iter=1000,
    penalty='l1', solver='liblinear', C=0.001,
).fit(recount_embeddings.query('cluster.isin([1, 2])')[fts].dropna(), recount_embeddings.query('cluster.isin([1, 2])').dropna()['cluster'])

In [ ]:
imp = pd.DataFrame(clf.coef_, columns=fts).mean().sort_values().reset_index()
imp.columns = ['variable', 'score']

imp.query('score != 0')

In [ ]:
df['cluster_prime'] = clf.predict(df[fts])

In [ ]:
df['cluster_prime'] = clf.predict(df[fts])
df.query('cluster.isin([1, 2])').value_counts(['cluster', 'cluster_prime']).reset_index()

In [ ]:
import matplotlib.patches as patches

left_nodes = imp.query('score != 0').variable.values

f, axs = subplots(cols=4, rows=1, w=20, h=3, return_f=True)
for ix, node in enumerate([0, 1, 2, 3]):
    edges = att[att.variable.isin(left_nodes) & att.cluster.isin([node])][['variable', 'cluster', 'value', 'color']].values
    bipartite_graph(
        left_nodes, [node], edges, {}, ax=axs[ix], alpha = 1, pos_color='white', label_color='black',
        rename={},
    )